In [1]:
chpt_path = '/text-mol/Mol-LLMstep=39800-train_total_loss=0.108.ckpt'

In [49]:
import os
import torch
import warnings
import pytorch_lightning as pl
from model.blip2_stage3 import Blip2Stage3
import hydra
from omegaconf import OmegaConf, DictConfig


os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def flatten_dictconfig(config: DictConfig) -> DictConfig:
    """
    Flatten a nested DictConfig into a single level DictConfig with keys as the path to the original keys.

    Args:
    - config (DictConfig): The nested DictConfig to be flattened.
    - parent_key (str, optional): The base key to use for prefixing the keys. Defaults to ''.
    - separator (str, optional): The separator to use between keys. Defaults to '.'.

    Returns:
    - DictConfig: The flattened configuration.
    """

    # only flatten just first level
    items = []
    for k, v in config.items():
        new_key = k
        if isinstance(v, DictConfig):
            for kk, vv in v.items():
                items.append((f"{kk}", vv))
        else:
            items.append((new_key, v))
    return OmegaConf.create(dict(items))


# Example usage:
config_dict = {
    'selfies_token_path': '/text-mol/Mol-LLM/model/selfies_dict.txt', 
    'llm_model': 'mistralai/Mistral-7B-Instruct-v0.3', 
    'tasks': None, 
    'gin_hidden_dim': 300, 
    'gin_num_layers': 5, 
    'drop_ratio': 0.0, 
    'tune_gnn': False, 
    'used_gnn_layer': -1, 
    'gnn_jk': 'last', 
    'bert_hidden_dim': 768, 
    'bert_name': 'scibert', 
    'cross_attention_freq': 2, 
    'num_query_token': 32, 
    'bert_num_hidden_layers': 5, 
    'tune_llm': 'lora', 
    'peft_config': None, 
    'peft_dir': '', 
    'load_in_8bit': False, 
    'lora_r': 64, 
    'lora_alpha': 32, 
    'lora_dropout': 0.1, 
    'add_selfies_tokens': True, 
    'ckpt_path': None,
    'prompt': '[START_I_SMILES]{}[END_I_SMILES]', 
    'num_beams': 1, 
    'mol_representation': 'string_only', 
    'strategy_name': None, 
    'accelerator': 'gpu', 
    'devices': '0', 
    'precision': 'bf16-mixed', 
    'max_steps': -1, 
    'max_epochs': 12, 
    'second_stage_start_step': 1000, 
    'every_n_train_steps': 100, 
    'task': None, 
    'save_top_k': 10, 
    'llava_style': 0, 
    'num_workers': 0, 
    'skip_sanity_check': False, 
    'total_batch_size': 1024, 
    'batch_size': 11, 
    'inference_batch_size': 22, 
    'truncation': 1, 
    'padding': 'max_length', 
    'max_length': 512, 
    'inference_max_length': 512, 
    'gen_max_len': 256, 
    'min_len': 8, 
    'apply_sequence_packing': False, 
    'max_packing_size': -1, 
    'weight_decay': 0.05, 
    'warmup_lr': 1e-05, 
    'warmup_steps': 100, 
    'scheduler': 'linear_warmup_cosine_lr', 
    'optimizer': 'adamw', 
    'log_every_n_steps': 50, 
    'mol_string_randomization_ratio': -1, 
    'val_check_interval': 0.5, 
    'check_val_every_n_epoch': 1, 
    'trainset_resize': -1, 
    'valset_resize': -1, 
    'testset_resize': -1, 
    'test_on_trainset': False, 
    'filename': 'debugging', 
    'seed': 42, 
    'mode': 'ft', 
    'wandb_entity': 'mol-llm', 
    'wandb_project': 'mol-llm', 
    'wandb_log_freq': 100, 
    'wandb_id': None, 
}

# convert config to OmegaConf
config = flatten_dictconfig(config_dict)

pl.seed_everything(config.seed)


model = Blip2Stage3(config)

print("total params:", sum(p.numel() for p in model.parameters()))

ckpt_path = '/text-mol/Mol-LLM/step=39800-train_total_loss=0.108.ckpt'
ckpt = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=False)


Seed set to 42


loading file tokenizer.model from cache at /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/e0bc86c23ce5aae1db576c8cca6f06f1f73af2db/tokenizer.model
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/e0bc86c23ce5aae1db576c8cca6f06f1f73af2db/special_tokens_map.json
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/e0bc86c23ce5aae1db576c8cca6f06f1f73af2db/tokenizer_config.json
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/e0bc86c23ce5aae1db576c8cca6f06f1f73af2db/tokenizer.json


Added 2944 selfies tokens to the tokenizer


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/e0bc86c23ce5aae1db576c8cca6f06f1f73af2db/config.json
Model config MistralConfig {
  "architectures": [
    "MistralForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 32768,
  "model_type": "mistral",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "rms_norm_eps": 1e-05,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.44.2",
  "use_cache": true,
  "vocab_size": 32768
}

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.3/snapshots/e0bc86c23ce5aae1db576c8cca6f06f1

trainable params: 167,772,160 || all params: 7,440,183,296 || trainable%: 2.2549


loading file vocab.txt from cache at /root/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1/vocab.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading file tokenizer.json from cache at None
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1/config.json
Model config BertConfig {
  "_name_or_path": "allenai/scibert_scivocab_uncased",
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token

total params: 7440183296


_IncompatibleKeys(missing_keys=['blip2model.llm_model.base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.0.self_attn.o_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.0.mlp.gate_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.0.mlp.up_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.0.mlp.down_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.0.input_layernorm.weight', 'blip2model.llm_model.base_model.model.model.layers.0.post_attention_layernorm.weight', 'blip2model.llm_model.base_model.model.model.layers.1.self_attn.q_proj.base_layer.weight', 'blip2model.llm_model.base_model.model.model.layers.1.self_attn.k_proj.base_layer.we

In [41]:
import re
def generate_and_tokenize_prompt(data_point, tokenizer, max_length, test=False):
    default_system_prompt = 'You are a helpful assistant for molecular chemistry, \
to address tasks including molecular property classification, \
molecular property regression, chemical reaction prediction, \
molecule captioning, molecule generation. \n\n'

    input_text = "[INST] "  + default_system_prompt + data_point['input'] + " [/INST] "
    output_text = data_point['output']
    
    input_mol_string_pattern = re.compile(
            "<SELFIES>"
            + ".*?"
            + "</SELFIES>"
        )

    prompt_tokenized = tokenizer(
                            tokenizer.bos_token + input_text,
                            truncation=True,
                            max_length=max_length,
                            padding=False,
                            return_tensors=None,
                            add_special_tokens=False,
                            )
    target_tokenized = tokenizer(
                            output_text + ' ' + tokenizer.eos_token,
                            truncation=True,
                            max_length=max_length - len(prompt_tokenized['input_ids']),
                            padding=False,
                            return_tensors=None,
                            add_special_tokens=False,
                            )
    
    labels = target_tokenized['input_ids']
    
    tokenized_result = {
        'prompt_tokenized': prompt_tokenized,
        'target_tokenized': target_tokenized,
        'labels': labels,
        
    }
    
    return tokenized_result

In [42]:
tokenizer = model.blip2model.llm_tokenizer

In [43]:
example = {
    'input': 'predict the boiling point of the following molecule: C1=CC=CC=C1',
    'output': 'The boiling point of the molecule is 69.0 degrees Celsius.'
}

tokenized_example = generate_and_tokenize_prompt(example, tokenizer, config.max_length)

In [44]:
def data_collate_fn(batch, tokenizer, max_length, padding, pad_to_multiple_of=None, return_tensors=None, device=None):

        prompt_tokenized = [sample['prompt_tokenized'] for sample in batch]
        target_tokenized = [sample['target_tokenized'] for sample in batch]
        
        full_input_ids = [p['input_ids'] + t['input_ids'] for p, t in zip(prompt_tokenized, target_tokenized)]
        full_attention_mask = [p['attention_mask'] + t['attention_mask'] for p, t in zip(prompt_tokenized, target_tokenized)]
        
        full_input_ids = [f_ids[:max_length] for f_ids in full_input_ids]
        full_attention_mask = [f_ids[:max_length] for f_ids in full_attention_mask]
        
        features = tokenizer.pad(
            {'input_ids': full_input_ids, 'attention_mask': full_attention_mask},
            
            padding=padding,
            pad_to_multiple_of=pad_to_multiple_of,
            return_tensors=return_tensors,
        )
        
        prompt_features = tokenizer.pad(
            {'input_ids': [p['input_ids'] for p in prompt_tokenized], 'attention_mask': [p['attention_mask'] for p in prompt_tokenized]},
            padding=padding,
            pad_to_multiple_of=pad_to_multiple_of,
            return_tensors=return_tensors,
        )
        
        features['prompt_input_ids'] = prompt_features.input_ids  # ['input_ids']
        features['prompt_attention_mask'] = prompt_features.attention_mask  # ['attention_mask']            
 
        if tokenizer.padding_side == 'right':
            raise NotImplementedError('padding_side should be left')
      
        assert features.input_ids.size(1) <= max_length, f"features.input_ids.size(1)={features.input_ids.size(1)} > max_length={max_length}"

        if device is not None:
              for k, v in features.items():
                  features[k] = v.to(device)

        return features

In [45]:
batch = [tokenized_example] * 2
collated_batch = data_collate_fn(batch, tokenizer, config.max_length, padding='max_length', pad_to_multiple_of=128, return_tensors='pt', device='cuda:0')

Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.


In [46]:
model.blip2model.to('cuda:0')

Blip2Mistral(
  (llm_model): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): MistralForCausalLM_custom(
        (model): MistralModel(
          (embed_tokens): Embedding(35745, 4096)
          (layers): ModuleList(
            (0-31): 32 x MistralDecoderLayer(
              (self_attn): MistralSdpaAttention(
                (q_proj): lora.Linear(
                  (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=4096, out_features=64, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=64, out_features=4096, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): ParameterDict()
                  (

In [47]:
outputs = model.blip2model.generate(
    graphs=None,
    # input_tokens=prompt_tokens,
    input_ids=collated_batch.prompt_input_ids,
    attention_mask=collated_batch.prompt_attention_mask,
    is_mol_token=None,
    
    num_beams=config.num_beams,
    max_length=config.gen_max_len,
    min_length=config.min_len,
)

/miniconda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


In [48]:
outputs

GenerateDecoderOnlyOutput(sequences=tensor([[29473, 35743, 29473, 34461, 33600, 34461, 33600, 34461, 33600, 33094,
         35174, 29473, 35744, 29473,     2],
        [29473, 35743, 29473, 34461, 33600, 34461, 33600, 34461, 33600, 33094,
         35174, 29473, 35744, 29473,     2]], device='cuda:0'), scores=(tensor([[-10.6250, -10.4375,     -inf,  ...,  -3.0938,   6.2500,  -4.1562],
        [-10.3750, -10.2500,     -inf,  ...,  -2.5781,   5.8438,   0.0109]],
       device='cuda:0'), tensor([[-4.4688, -4.5625,    -inf,  ...,  3.5625, 52.7500,  5.3750],
        [-4.0938, -4.3438,    -inf,  ...,  4.6562, 48.7500,  7.5938]],
       device='cuda:0'), tensor([[-10.6250, -10.4375,     -inf,  ...,   0.1099,   9.5625,  -7.3750],
        [-11.0625, -10.8750,     -inf,  ...,  -1.4141,   5.9688,  -6.9688]],
       device='cuda:0'), tensor([[-7.5000, -7.0000,    -inf,  ...,  0.4844,  6.0000,  4.1250],
        [-7.4375, -6.9688,    -inf,  ...,  1.6016,  4.6875,  4.9375]],
       device='cuda:0'), t